In [ ]:
import numpy as np
import pandas as pd

# ========= 1) Excel 读列 =========
file_path = r"your_data.xlsx"
sheet_name = "SA"
df = pd.read_excel(file_path, sheet_name=sheet_name)

# ========= 2) 参数模板 =========
params = {
    "x0": np.array([5.0, 5.0]),         # 初始解
    "lb": np.array([-10.0, -10.0]),     # 下界
    "ub": np.array([10.0, 10.0]),       # 上界
    "T0": 100.0,                        # 初始温度
    "cooling_rate": 0.95,               # 降温系数
    "n_iter": 500,                      # 迭代次数
    "step_scale": 0.5                   # 邻域扰动尺度
}

def obj(x): return np.sum((x-1)**2)

x = params["x0"].copy().astype(float)
fx = obj(x)
best_x, best_fx = x.copy(), fx
T = params["T0"]

for _ in range(params["n_iter"]):
    xn = np.clip(x + params["step_scale"]*np.random.randn(*x.shape), params["lb"], params["ub"])
    fn = obj(xn)
    if fn < fx or np.random.rand() < np.exp(-(fn - fx)/max(T, 1e-12)):
        x, fx = xn, fn
        if fx < best_fx:
            best_x, best_fx = x.copy(), fx
    T *= params["cooling_rate"]

print(best_x, best_fx)


In [ ]:
"""
模拟退火

使用方法：
1. 按照下方 TODO 修改 DATA_FILE、列名、参数和输出文件名。
2. 将数据文件放在本脚本同目录，或把 DATA_FILE 改成绝对路径。
3. 运行：python "模拟退火.py"
"""

from pathlib import Path
import numpy as np
import pandas as pd



DATA_FILE = "data.csv"  # TODO: 请填写[数据文件路径]，说明：CSV/Excel 均可；若使用 Excel，请在 load_data 中改为 read_excel。
OUTPUT_FILE = "model_output.csv"  # TODO: 请填写[输出文件名]，说明：保存模型结果，建议保留 .csv 或 .xlsx 后缀。
RANDOM_STATE = 42  # TODO: 请填写[随机种子]，说明：用于复现实验；整数即可。
INITIAL_X = [0.0, 0.0]  # TODO: 请填写[初始解]，说明：长度等于变量数。
LOWER_BOUND = -5  # TODO: 请填写[变量下界]，说明：可扩展为数组。
UPPER_BOUND = 5  # TODO: 请填写[变量上界]，说明：必须大于下界。
INITIAL_TEMP = 100.0  # TODO: 请填写[初始温度]，说明：越大越容易接受差解。
FINAL_TEMP = 1e-3  # TODO: 请填写[终止温度]，说明：小于该温度停止。
COOLING_RATE = 0.95  # TODO: 请填写[降温系数]，说明：0 到 1，越接近 1 越慢。
STEP_SCALE = 0.2  # TODO: 请填写[扰动尺度]，说明：控制邻域搜索范围。
TARGET_VECTOR = np.array([1.0, 2.0])  # TODO: 请填写[示例目标向量]，说明：请替换为真实目标函数参数。



REQUIRES_DATA = False  # 参数型模型可不提供数据文件；表格型模型必须提供数据。


def load_data() -> pd.DataFrame:
    """读取用户数据；竞赛时通常把 Excel/CSV 表格整理成一行一个样本。"""
    path = Path(DATA_FILE)
    if not path.exists():
        if not REQUIRES_DATA:
            return pd.DataFrame()
        raise FileNotFoundError(
            f"未找到数据文件 {DATA_FILE}。请先修改 DATA_FILE，或将数据放到脚本同目录。"
        )
    if path.suffix.lower() in [".xlsx", ".xls"]:
        return pd.read_excel(path)
    return pd.read_csv(path)


def objective(x):
    return np.sum((x - TARGET_VECTOR) ** 2)


def run_model(data: pd.DataFrame) -> None:
    rng = np.random.default_rng(RANDOM_STATE)
    x = np.array(INITIAL_X, dtype=float)
    best = x.copy()
    temp = INITIAL_TEMP
    while temp > FINAL_TEMP:
        candidate = np.clip(x + rng.normal(0, STEP_SCALE, size=x.shape), LOWER_BOUND, UPPER_BOUND)
        delta = objective(candidate) - objective(x)
        if delta < 0 or rng.random() < np.exp(-delta / temp):
            x = candidate
        if objective(x) < objective(best):
            best = x.copy()
        temp *= COOLING_RATE
    print("最优解近似:", best, "目标值:", objective(best))


if __name__ == "__main__":
    df = load_data()
    run_model(df)
